# Explainable decision-maker backtest

This notebook invokes the production TypeScript decision-maker on a chronological period. It evaluates directional signal quality on the underlying asset; it does **not** simulate option P&L or submit orders.

## Period definitions

- `TIMEFRAME` controls the market-bar size.
- `LOOKBACK_BARS` controls how much history the agent sees at each timestamp.
- `EVALUATION_STEP_BARS` controls how frequently it runs.
- `FORWARD_HORIZONS` controls when direction is judged, in future bars.

Use a short one-week period only as a smoke test. Use at least 18–24 months of daily bars for meaningful evaluation.

In [ ]:
# Install only the notebook's plotting/table dependencies into this Python kernel.
import subprocess
import sys
from pathlib import Path

requirements_candidates = [
    Path.cwd() / 'research' / 'decision-maker' / 'requirements.txt',
    Path.cwd() / 'requirements.txt',
]
requirements_path = next((path for path in requirements_candidates if path.exists()), None)
if requirements_path is None:
    raise RuntimeError('Could not locate research/decision-maker/requirements.txt')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(requirements_path)],
    check=True,
)

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'package.json').exists():
            return candidate
    raise RuntimeError('Could not find the project root containing package.json')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
NPM = shutil.which('npm') or shutil.which('npm.cmd')
if not NPM:
    raise RuntimeError('npm is not available in this notebook environment')
OUTPUT_DIR = PROJECT_ROOT / 'research' / 'decision-maker' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project: {PROJECT_ROOT}')

In [ ]:
# Edit this cell to choose the experiment.
SYMBOLS = ['SPY', 'QQQ', 'GLD']
START = '2024-02-01'
END = '2026-07-31'
TIMEFRAME = '1Day'       # 1Day, 1Hour, or 15Min
LOOKBACK_BARS = 100
EVALUATION_STEP_BARS = 1
FORWARD_HORIZONS = [1, 3, 5]
DATA_FEED = 'iex'        # Works with Alpaca paper/free market-data access

assert TIMEFRAME in {'1Day', '1Hour', '15Min'}
assert LOOKBACK_BARS >= 50
assert all(horizon > 0 for horizon in FORWARD_HORIZONS)

In [ ]:
def run_period(symbol: str) -> dict:
    output_path = OUTPUT_DIR / f'{symbol.lower()}-{TIMEFRAME.lower()}-{START}-{END}.json'
    command = [
        NPM, 'run', 'decision:backtest', '--',
        '--symbol', symbol,
        '--start', START,
        '--end', END,
        '--timeframe', TIMEFRAME,
        '--lookback', str(LOOKBACK_BARS),
        '--step', str(EVALUATION_STEP_BARS),
        '--horizons', ','.join(map(str, FORWARD_HORIZONS)),
        '--output', str(output_path),
    ]
    if DATA_FEED:
        command.extend(['--feed', DATA_FEED])
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, capture_output=True, text=True
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or completed.stdout.strip())
    return json.loads(output_path.read_text(encoding='utf-8'))

# This performs read-only Alpaca market-data requests.
reports = {symbol: run_period(symbol) for symbol in SYMBOLS}
print(f'Completed {len(reports)} symbol backtests.')

## Coverage and directional accuracy

Coverage measures how often the signal exceeded the configured trade threshold. Accuracy is shown for every requested forward horizon. Balanced accuracy prevents an always-bullish strategy from benefiting only from market bias.

In [ ]:
summary_rows = []
for symbol, report in reports.items():
    summary = report['summary']
    for horizon in summary['horizons']:
        summary_rows.append({
            'symbol': symbol,
            'horizon_bars': horizon['horizonBars'],
            'evaluations': summary['evaluations'],
            'eligible_signals': summary['eligibleSignals'],
            'coverage': summary['coverage'],
            'directional_accuracy': horizon['directionalAccuracy'],
            'balanced_accuracy': horizon['balancedAccuracy'],
            'average_signed_return': horizon['averageSignedReturn'],
            'median_signed_return': horizon['medianSignedReturn'],
            'non_overlapping_accuracy': horizon['nonOverlappingAccuracy'],
            'max_consecutive_incorrect': summary['maximumConsecutiveIncorrect'],
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.style.format({
    'coverage': '{:.1%}',
    'directional_accuracy': '{:.1%}',
    'balanced_accuracy': '{:.1%}',
    'average_signed_return': '{:.3%}',
    'median_signed_return': '{:.3%}',
    'non_overlapping_accuracy': '{:.1%}',
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
coverage = summary_df.groupby('symbol', as_index=True)['coverage'].first()
coverage.plot(kind='bar', ax=axes[0], color='#34d399', title='Signal coverage')
axes[0].set_ylabel('Eligible signals / evaluations')
axes[0].set_ylim(0, 1)
accuracy = summary_df.pivot(index='symbol', columns='horizon_bars', values='directional_accuracy')
accuracy.plot(kind='bar', ax=axes[1], title='Directional accuracy by horizon')
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=1)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Accuracy')
plt.tight_layout()

## Inspect signal strength and one decision

A stronger score should generally correspond to better forward results. The final cell exposes the exact features and strategy contributions for the strongest eligible decision.

In [ ]:
primary_horizon = str(max(FORWARD_HORIZONS))
observation_rows = []
for symbol, report in reports.items():
    for item in report['observations']:
        outcome = item['outcomes'].get(primary_horizon)
        observation_rows.append({
            'symbol': symbol,
            'as_of': item['asOf'],
            'eligible': item['eligible'],
            'direction': item['direction'],
            'regime': item['regime'],
            'score': item['finalScore'],
            'correct': None if outcome is None else outcome['correct'],
            'signed_return': None if outcome is None else outcome['signedReturn'],
            'blocking_reason': item['blockingReason'],
        })
observations_df = pd.DataFrame(observation_rows)
eligible_df = observations_df[observations_df['eligible']].copy()
eligible_df['absolute_score'] = eligible_df['score'].abs()
eligible_df.sort_values('absolute_score', ascending=False).head(20)

In [ ]:
if eligible_df.empty:
    print('No eligible signals were produced. Inspect no-trade reasons and adjust only on the development period.')
else:
    strongest = eligible_df.sort_values('absolute_score', ascending=False).iloc[0]
    source_report = reports[strongest['symbol']]
    decision = next(item for item in source_report['observations'] if item['asOf'] == strongest['as_of'])
    display(pd.Series({
        'symbol': strongest['symbol'],
        'as_of': decision['asOf'],
        'direction': decision['direction'],
        'score': decision['finalScore'],
        'regime': decision['regime'],
        'data_warnings': decision['dataQuality']['warnings'],
    }))
    display(pd.Series(decision['features'], name='feature_value').to_frame())
    display(pd.DataFrame(decision['contributions']))

## Interpretation guardrails

- Accuracy near 50% is not evidence of useful direction.
- Prefer balanced and non-overlapping accuracy over raw accuracy alone.
- Positive average signed return should persist across symbols, regimes, and months.
- Tune thresholds only on a development interval, then evaluate once on an untouched holdout interval.
- These results exclude option spreads, slippage, exits, sizing, and risk-manager decisions, so they are not portfolio-profit claims.